# Summary — Note Summarisation

Runs summarisation across the vault using both backends (Ollama and Claude) and compares
results against the ground truth in `data/ground_truth.yaml`.

Unlike NER and tags, summaries are compared by reading — there is no exact-match metric.

In [ ]:
import sys
sys.path.insert(0, '../src')

from glob import glob
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from notemine.config import load_config
from notemine.frontmatter_io import load_note
from notemine.ground_truth import load_ground_truth
from notemine.tasks import run_summary

load_dotenv('../.env')
config = load_config('../config.toml')
gt = load_ground_truth('../' + config['vault']['ground_truth'].lstrip('./'))

vault = '../' + config['vault']['path'].lstrip('./')
config['vault']['prompts_dir'] = '../' + config['vault']['prompts_dir'].lstrip('./')

notes = sorted(
    p for p in glob(f'{vault}/**/*.md', recursive=True)
    if not Path(p).name.startswith('index_')
)
print(f'{len(notes)} notes found')

## Run summarisation with both backends

In [ ]:
BACKENDS = ['ollama', 'claude']

results = []
for path in notes:
    name = Path(path).name
    post = load_note(path)
    if len(post.content.strip()) < 50:
        continue

    gt_entry = gt.get(name, {})

    row = {
        'file': name,
        'lang': gt_entry.get('lang', '?'),
        'ground_truth': gt_entry.get('summary', ''),
    }

    for backend in BACKENDS:
        try:
            sentences = run_summary(backend, post.content, config)
            row[backend] = ' '.join(sentences)
        except Exception as e:
            row[backend] = f'ERROR: {e}'

    results.append(row)
    print(f'  {name}')

print(f'\nDone — {len(results)} notes processed')

## Side-by-side comparison table

Read each row to compare ground truth vs both backends.

In [ ]:
pd.set_option('display.max_colwidth', 120)
df = pd.DataFrame([{
    'file': r['file'],
    'lang': r['lang'],
    'ground_truth': r['ground_truth'],
    'ollama': r['ollama'],
    'claude': r['claude'],
} for r in results])
df

## Inspect a single note

Change `INSPECT` to read one note's summaries in full.

In [ ]:
INSPECT = Path(notes[0]).name

row = next((r for r in results if r['file'] == INSPECT), None)
if row:
    print(f'=== {INSPECT} ===\n')
    print('GROUND TRUTH:')
    print(row['ground_truth'])
    print(f'\nOLLAMA ({config["ollama"]["model"]}):')
    print(row['ollama'])
    print(f'\nCLAUDE ({config["claude"]["model"]}):')
    print(row['claude'])

## (Optional) Save results to frontmatter

Uncomment to write results under `notemine.summary.ollama` / `notemine.summary.claude`.

In [ ]:
# from notemine.frontmatter_io import save_note
#
# for path in notes:
#     name = Path(path).name
#     row = next((r for r in results if r['file'] == name), None)
#     if row is None:
#         continue
#     post = load_note(path)
#     notemine = post.metadata.get('notemine', {})
#     summary_data = notemine.get('summary', {})
#     if not row.get('ollama', '').startswith('ERROR'):
#         summary_data['ollama'] = row['ollama']
#     if not row.get('claude', '').startswith('ERROR'):
#         summary_data['claude'] = row['claude']
#     notemine['summary'] = summary_data
#     post.metadata['notemine'] = notemine
#     save_note(path, post)
# print('Saved.')